# 03 — Análisis CAD/DXF para piloto MT Montecarlo

Esta notebook revisa las fuentes CAD originales con prioridad en `dxf/Montecarlo_MT.dxf`. El objetivo es identificar capas útiles para el primer piloto PostGIS de media tensión, sin convertir todavía los archivos.


## 1. Objetivo de esta notebook

El objetivo no es cargar datos todavía. Primero hacemos una exploración segura y de solo lectura:

- Inventariar archivos CAD/DXF disponibles.
- Priorizar `Montecarlo_MT.dxf` como muestra del piloto.
- Confirmar capas MT candidatas.
- Separar fases posteriores: BT/suministros, catastro, ejes viales y otros dominios.


## 2. Preparación

Ubicamos la raíz del proyecto y las carpetas originales. No vamos a escribir ni modificar archivos dentro de `dwg/` o `dxf/`.


In [1]:
# pathlib permite trabajar con rutas de forma clara y portable.
from pathlib import Path

# pandas nos ayuda a presentar inventarios y resúmenes como tablas.
import pandas as pd

# Buscamos la raíz del proyecto para que el notebook sea ejecutable desde Jupyter.
RAIZ = Path.cwd()
while RAIZ.name != 'ceml_gis' and RAIZ.parent != RAIZ:
    RAIZ = RAIZ.parent

# Definimos carpetas de fuentes CAD originales.
DXF_DIR = RAIZ / 'dxf'
DWG_DIR = RAIZ / 'dwg'
DXF_MT = DXF_DIR / 'Montecarlo_MT.dxf'

print('Carpeta DXF:', DXF_DIR)
print('Carpeta DWG:', DWG_DIR)
print('DXF MT existe:', DXF_MT.exists())


Carpeta DXF: /home/martin/Code/ceml_gis/dxf
Carpeta DWG: /home/martin/Code/ceml_gis/dwg
DXF MT existe: True


## 3. Mapa de fuentes CAD

Este mapa evita mezclar alcances:

- `Montecarlo_MT.dxf` / `Montecarlo_MT.dwg`: red MT, postes, elementos, seccionadores, SET y textos de referencia.
- `Montecarlo_SUMINISTRO_Y_BT.dwg`: BT, suministros, medidores y luminarias; no hay DXF equivalente disponible en esta carpeta.
- `MONTECARLO_CATASTROORIGINAL.dxf` / `.dwg`: catastro, calles, `ejes_viales`, árboles y referencias urbanas.

Caveat de catastro: las parcelas LWPOLYLINE no se unen directamente por `idcad` porque esas filas no traen `IDCAD/COOP`; para unir atributos se requiere usar etiquetas de `.00.Urbano_Texto_Parcelas` y contención espacial.


In [8]:
# Esta función convierte tamaños en bytes a una unidad fácil de leer en clase.
def formato_tamano(bytes_: int) -> str:
    # Mostramos tamaños legibles para comparar rápidamente archivos grandes y pequeños.
    for unidad in ['B', 'KB', 'MB', 'GB']:
        if bytes_ < 1024:
            return f'{bytes_:.1f} {unidad}'
        bytes_ /= 1024
    return f'{bytes_:.1f} TB'

# Inventariamos fuentes CAD sin abrir ni modificar su contenido.
filas = []
for carpeta, tipo in [(DXF_DIR, 'DXF'), (DWG_DIR, 'DWG')]:
    for path in sorted(carpeta.glob('*')):
        filas.append({'archivo': path.name, 'tipo': tipo, 'tamaño': formato_tamano(path.stat().st_size)})

inventario_cad = pd.DataFrame(filas)
inventario_cad


,archivo,tipo,tamaño
0,MONTECARLO_CATASTROORIGINAL.dxf,DXF,97.7 MB
1,Montecarlo_MT.dxf,DXF,95.1 MB
2,MONTECARLO_CATASTROORIGINAL.dwg,DWG,36.2 MB
3,Montecarlo_MT.dwg,DWG,35.9 MB
4,Montecarlo_SUMINISTRO_Y_BT.dwg,DWG,16.1 MB


## 4. Capas MT objetivo

Para el primer piloto PostGIS, la interpretación de capas es concreta:

| Capa DXF | Geometría destino | Uso inicial |
|---|---|---|
| `.00.MT_Cables` | `LINESTRING` | trazas de cables MT |
| `.00.MT_Postes` | `POINT` | postes MT |
| `.00.MT_Elementos` | `POINT` | elementos de red MT |
| `.00.MT_Seccionadores` | `POINT` | seccionadores |
| `.00.SET` | `POINT` | SET |
| `.00.SET_Textos` | referencia/texto | etiquetas para control y validación |

Estas capas son el alcance inicial. BT/suministros y catastro son fases posteriores.


In [3]:
# Organizamos el mapeo de capas MT hacia capas GIS destino.
# Esta tabla todavía no convierte geometrías; solo define el criterio del piloto.
capas_mt = pd.DataFrame([
    {'capa_dxf': '.00.MT_Cables', 'geometria': 'LINESTRING', 'capa_postgis': 'gis.mt_cables'},
    {'capa_dxf': '.00.MT_Postes', 'geometria': 'POINT', 'capa_postgis': 'gis.mt_postes'},
    {'capa_dxf': '.00.MT_Elementos', 'geometria': 'POINT', 'capa_postgis': 'gis.mt_elementos'},
    {'capa_dxf': '.00.MT_Seccionadores', 'geometria': 'POINT', 'capa_postgis': 'gis.mt_seccionadores'},
    {'capa_dxf': '.00.SET', 'geometria': 'POINT', 'capa_postgis': 'gis.set'},
    {'capa_dxf': '.00.SET_Textos', 'geometria': 'TEXT', 'capa_postgis': 'referencia para validación'},
])
capas_mt


,capa_dxf,geometria,capa_postgis
0,.00.MT_Cables,LINESTRING,gis.mt_cables
1,.00.MT_Postes,POINT,gis.mt_postes
2,.00.MT_Elementos,POINT,gis.mt_elementos
3,.00.MT_Seccionadores,POINT,gis.mt_seccionadores
4,.00.SET,POINT,gis.set
5,.00.SET_Textos,TEXT,referencia para validación


## 5. Vista rápida de estructura DXF sin dependencias pesadas

Un DXF suele organizarse en pares de líneas llamados *group code* y *value*. Esta vista rápida no reemplaza a un parser CAD, pero permite enseñar cómo aparecen entidades, capas y valores.


In [4]:
# Buscamos solamente archivos DXF para confirmar que Montecarlo_MT.dxf está disponible.
# Priorizamos el archivo MT porque es el insumo del primer piloto.
archivos_dxf = sorted(DXF_DIR.glob('*.dxf'))
print('DXF disponibles:')
for path in archivos_dxf:
    marca = ' <- muestra prioritaria MT' if path.name == 'Montecarlo_MT.dxf' else ''
    print(f'- {path.name}{marca}')


DXF disponibles:
- MONTECARLO_CATASTROORIGINAL.dxf
- Montecarlo_MT.dxf <- muestra prioritaria MT


In [5]:
# Leemos un fragmento del DXF MT como texto para observar su estructura básica.
# errors='replace' evita cortar la clase si aparece algún byte no decodificable.
def leer_lineas_dxf(path: Path, limite: int = 80) -> list[str]:
    # Recorremos hasta el límite solicitado sin fallar si el archivo tuviera menos líneas.
    lineas = []
    with path.open('r', encoding='utf-8', errors='replace') as archivo:
        for numero, linea in enumerate(archivo):
            if numero >= limite:
                break
            lineas.append(linea.rstrip('\n'))
    return lineas

if DXF_MT.exists():
    for numero, linea in enumerate(leer_lineas_dxf(DXF_MT, limite=80), start=1):
        print(f'{numero:03d}: {linea}')
else:
    print('No se encontró dxf/Montecarlo_MT.dxf.')


001:   0
002: SECTION
003:   2
004: HEADER
005:   9
006: $ACADVER
007:   1
008: AC1027
009:   9
010: $ACADMAINTVER
011:  70
012:    105
013:   9
014: $DWGCODEPAGE
015:   3
016: ANSI_1252
017:   9
018: $LASTSAVEDBY
019:   1
020: martin
021:   9
022: $REQUIREDVERSIONS
023: 160
024:                  0
025:   9
026: $INSBASE
027:  10
028: 0.0
029:  20
030: 0.0
031:  30
032: 0.0
033:   9
034: $EXTMIN
035:  10
036: 0.0
037:  20
038: 0.0
039:  30
040: -443.6501915965321
041:   9
042: $EXTMAX
043:  10
044: 780884.6892474203
045:  20
046: 7074558.815062655
047:  30
048: 762.12525055
049:   9
050: $LIMMIN
051:  10
052: 0.0
053:  20
054: 0.0
055:   9
056: $LIMMAX
057:  10
058: 420.0
059:  20
060: 297.0
061:   9
062: $ORTHOMODE
063:  70
064:      0
065:   9
066: $REGENMODE
067:  70
068:      1
069:   9
070: $FILLMODE
071:  70
072:      1
073:   9
074: $QTEXTMODE
075:  70
076:      0
077:   9
078: $MIRRTEXT
079:  70
080:      0


## 6. Conteo simple de entidades por capa

Esta celda hace una lectura liviana de pares DXF para contar entidades por capa. No reemplaza a una conversión con parser especializado, pero ayuda a confirmar si las capas MT objetivo aparecen en el archivo.


In [6]:
# Definimos entidades CAD que suelen tener geometría o referencia espacial.
# El conteo se limita al DXF MT para mantener el análisis enfocado.
ENTIDADES_CAD = {'LINE', 'LWPOLYLINE', 'INSERT', 'CIRCLE', 'TEXT', 'MTEXT', 'POINT'}

# Parseamos pares group-code/value de forma simple para contar entidad y capa.
def contar_entidades_por_capa(path: Path) -> pd.DataFrame:
    # Leemos todas las líneas porque necesitamos observar pares sucesivos del DXF.
    lineas = path.read_text(encoding='utf-8', errors='replace').splitlines()
    filas = []
    entidad_actual = None
    capa_actual = None
    indice = 0
    while indice + 1 < len(lineas):
        codigo = lineas[indice].strip()
        valor = lineas[indice + 1].strip()
        if codigo == '0':
            if entidad_actual and capa_actual:
                filas.append({'entidad': entidad_actual, 'capa': capa_actual})
            entidad_actual = valor if valor in ENTIDADES_CAD else None
            capa_actual = None
        elif codigo == '8' and entidad_actual:
            capa_actual = valor
        indice += 2
    if entidad_actual and capa_actual:
        filas.append({'entidad': entidad_actual, 'capa': capa_actual})
    if not filas:
        return pd.DataFrame(columns=['entidad', 'capa', 'cantidad'])
    return pd.DataFrame(filas).value_counts(['entidad', 'capa']).reset_index(name='cantidad')

if DXF_MT.exists():
    conteo_mt = contar_entidades_por_capa(DXF_MT)
    capas_objetivo = set(capas_mt['capa_dxf'])
    conteo_mt[conteo_mt['capa'].isin(capas_objetivo)].sort_values(['capa', 'entidad'])
else:
    print('No se puede contar entidades porque falta Montecarlo_MT.dxf.')


## 7. Criterios de prioridad

La prioridad de migración queda ordenada así:

1. Piloto MT Montecarlo: capas `.00.MT_*` y `.00.SET*` del DXF MT, enriquecidas con tablas SQL Server del piloto.
2. BT/suministros: fuente `Montecarlo_SUMINISTRO_Y_BT.dwg`, sin DXF equivalente disponible en esta etapa.
3. Catastro y contexto: `MONTECARLO_CATASTROORIGINAL.dxf`, incluyendo `ejes_viales`, con uniones espaciales y etiquetas cuando corresponda.

No se debe iniciar con una migración amplia de todo GIS porque mezclaría dominios con reglas geométricas distintas.


In [7]:
# Esta matriz resume fases y evita ampliar el alcance sin control técnico.
# Se usa como guía para diseñar la carga PostGIS de la notebook 04.
criterios = pd.DataFrame([
    {'fase': 1, 'alcance': 'MT Montecarlo', 'fuente': 'Montecarlo_MT.dxf + tablas MT SQL Server', 'estado': 'piloto inicial'},
    {'fase': 2, 'alcance': 'BT y suministros', 'fuente': 'Montecarlo_SUMINISTRO_Y_BT.dwg', 'estado': 'posterior'},
    {'fase': 3, 'alcance': 'Catastro y ejes_viales', 'fuente': 'MONTECARLO_CATASTROORIGINAL.dxf', 'estado': 'posterior con caveat de etiquetas'},
])
criterios


,fase,alcance,fuente,estado
0,1,MT Montecarlo,Montecarlo_MT.dxf + tablas MT SQL Server,piloto inicial
1,2,BT y suministros,Montecarlo_SUMINISTRO_Y_BT.dwg,posterior
2,3,Catastro y ejes_viales,MONTECARLO_CATASTROORIGINAL.dxf,posterior con caveat de etiquetas


## 8. Próximos pasos

La notebook 04 debe diseñar una carga PostGIS en etapas: crudo, depuración, reconstrucción geométrica y publicación GIS. El diseño debe registrar rechazos y métricas, especialmente la cobertura de unión entre `Objetos_Red` y `TMP_SHAPEFILE`.
